# S6_04 — 리소스(Resources): 데이터를 노출하고 클라이언트에서 읽기

이번 노트북에서는 도구(Tools) 다음으로 MCP 서버의 두 번째 핵심 컴포넌트인 리소스를 학습한다. 강의노트의 두 절(§2.2 와 §2.3)을 차례로 코드로 따라가면서, 서버 측에서 리소스를 정의하는 방법과 클라이언트 측에서 그것을 읽어 오는 방법을 모두 익힌다. 두 절을 한 노트북에서 함께 다루는 이유는, 정의와 접근이 짝을 이루어야 비로소 학습의 의미가 완성되기 때문이다.

본 노트북이 매핑되는 Skilljar 레슨은 두 개다. 첫째는 Skilljar L07 (ID 287782, 리소스 정의하기 레슨)이고, 강의노트로는 §2.2 (라인 약 896 부터 1046 까지)에 대응한다. 둘째는 Skilljar L08 (ID 287783, 리소스 접근하기 레슨)이고, 강의노트로는 §2.3 (라인 약 1050 부터 1162 까지)에 대응한다.

**학습 목표**:

본 노트북을 마치면 학생은 다음 네 가지를 할 수 있어야 한다. 첫째, 리소스 데코레이터를 사용하여 두 가지 형태의 리소스를 모두 정의할 수 있다. 매개변수가 없는 고정된 형태의 리소스와 매개변수가 들어가는 형태의 리소스가 그것이다. 둘째, 마임(MIME) 타입을 명시적으로 지정함으로써 클라이언트에 응답 포맷이 무엇인지를 정확히 알려 줄 수 있다. 가장 자주 사용되는 두 가지는 자바스크립트 객체 표기법과 평문 텍스트 형태다. 셋째, 클라이언트 메서드를 사용하여 서버 리소스를 읽고, 응답의 마임 타입에 따라 적절히 분기 처리하는 로직을 작성할 수 있다. 넷째, 사용자가 골뱅이(`@`) 기호로 문서를 멘션하는 워크플로를 직접 시뮬레이션함으로써, 사용자의 의도가 어떻게 리소스 페치로 변환되고 최종 프롬프트에 주입되는지를 체험할 수 있다.

**선수 학습 사항**:

본 노트북을 시작하기 전에 몇 가지 사항이 갖춰져 있어야 한다. 우선 직전 노트북인 클라이언트 구축 노트북을 먼저 완료해야 한다. 그래야 같은 폴더에 서버 파일과 클라이언트 파일이 이미 생성되어 있는 상태가 된다. 또한 서버 파일 안에는 두 개의 도구가 정의되어 있어야 하고, 클라이언트 파일에는 클라이언트 클래스가 연결과 도구 목록 조회와 도구 호출과 정리 같은 메서드들을 갖춘 상태로 구현되어 있어야 한다.

**최종 산출물 약속**:

본 노트북이 종료될 시점에는 두 가지 결과물이 만들어진다. 첫째로 서버 파일이 도구 두 개와 리소스 두 개를 모두 포함한 완성본이 된다. 둘째로 클라이언트 파일에 리소스를 읽는 새로운 메서드가 추가되어, 클라이언트가 서버의 리소스를 직접 읽어 올 수 있는 상태가 된다. 이 두 결과물이 다음 노트북의 출발점이 된다.


## §1. 리소스란 무엇인가 — 도구와의 차이점

엠시피(MCP) 서버에서 리소스란 클라이언트에게 데이터를 노출하는 메커니즘을 가리킨다. 그 동작 방식은 웹 통신의 정보 조회 요청과 비슷하다고 보면 된다. 즉 어떤 동작을 능동적으로 수행하는 것이 아니라, 정보를 조회하는 시나리오에 적합한 컴포넌트라는 뜻이다. 강의노트 §2.2 의 핵심 발견 박스에서 정리한 표현을 그대로 옮기면 이렇다. 리소스는 데이터를 노출하는 역할을 맡고, 도구는 동작을 수행하는 역할을 맡는다는 구분이 가장 핵심이다.

두 컴포넌트가 어떻게 다른지 항목별로 풀어 보자. 우선 데코레이터부터 다르다. 도구를 정의할 때는 도구 데코레이터를 사용하지만, 리소스를 정의할 때는 리소스 데코레이터를 사용하며 이때 유알아이(URI)와 마임 타입을 함께 지정해야 한다. 다음으로 비유 측면에서 보자. 도구는 게시 요청이나 갱신 요청에 가까워서 상태를 변경하는 성격을 가지지만, 리소스는 조회 요청에 가까워서 부작용을 일으키지 않는다.

주요 용도 측면에서도 차이가 분명하다. 도구는 동작을 수행하거나, 상태를 변경하거나, 외부 시스템에 명령을 보내는 시나리오에 사용된다. 반면 리소스는 데이터를 단순히 노출하거나, 어떤 정보를 조회하거나, 부작용 없는 응답을 만드는 시나리오에 사용된다.

호출 주체 측면도 중요한 구분이 된다. 도구의 경우에는 클로드(Claude)가 능동적으로 호출을 결정한다. 즉 클로드가 사용자의 요청을 보고 어떤 도구를 써야겠다고 스스로 판단하여 도구 사용 이벤트를 발생시키는 것이다. 그러나 리소스의 경우에는 사용자나 애플리케이션이 명시적으로 선택한다. 가장 대표적인 예가 사용자가 골뱅이 기호로 문서를 멘션하는 워크플로다. 마지막으로 클라이언트 측 메서드 이름도 다르다. 도구를 호출할 때는 도구 호출 메서드에 도구 이름과 인자를 함께 전달하지만, 리소스를 읽을 때는 리소스 읽기 메서드에 유알아이만 전달하면 된다.

> [!finding] 핵심 설계 원칙 — 읽기와 변경의 구분
> 강의노트의 핵심 메시지를 한 문장으로 정리하면 이렇다. 읽기만 한다면 리소스로 만들고, 무언가를 변경한다면 도구로 만들어라. 이 원칙이 왜 중요한지 시나리오로 살펴보자. 사용자가 자기 의도를 골뱅이 기호로 이미 명시적으로 선언한 상황을 떠올려 보자. 만약 도구 호출 방식으로 이 워크플로를 구현했다면, 클로드가 한 번 더 문서를 읽어야겠다고 결정해야 하므로 언어모델 왕복이 한 번 추가된다. 그러나 리소스 방식으로 구현한다면, 첫 메시지에 문서 내용이 이미 포함된 채로 클로드에게 전달된다. 결과적으로 사용자의 의도가 명확한 시나리오에서는 리소스 방식이 훨씬 빠르고 효율적인 선택이 된다.

**리소스의 두 가지 종류**:

리소스는 유알아이의 형태에 따라 두 가지로 나뉜다. 첫 번째는 고정된 유알아이를 가지는 직접 리소스다. 이 형태는 매개변수가 전혀 없는 정적인 유알아이를 가진다. 강의 예제에서 사용하는 문서 목록 유알아이가 바로 여기에 해당한다. 두 번째는 매개변수가 들어가는 매개변수 리소스다. 이 형태는 유알아이 안에 매개변수 자리표시자가 중괄호로 표시되어 있다. 예를 들어 문서 아이디 자리표시자가 들어간 형태가 그것이다.

매개변수 유알아이의 경우, 파이썬 에스디케이(SDK)가 유알아이에서 매개변수 값을 자동으로 파싱하여 함수의 키워드 인자로 전달해 주는 점이 핵심이다. 예를 들어 클라이언트가 특정 문서 아이디가 들어간 유알아이로 요청을 보내면, 서버 측에서는 그 아이디 값이 함수에 자동으로 주입되어 호출된다. 다만 이 자동 매핑이 가능하려면 유알아이의 자리표시자 이름과 함수 매개변수 이름이 정확히 일치해야 한다는 점을 잊지 말아야 한다.


In [ ]:
# Setup — Week_07.md §2.2 line ~938 — FastMCP + base 임포트
from pydantic import Field
from mcp.server.fastmcp import FastMCP
from mcp.server.fastmcp.prompts import base
import os

print('FastMCP imported. Working dir:', os.getcwd())

## §2. 고정된 형태의 리소스 만들기 — 모든 문서 아이디 목록 반환

먼저 가장 단순한 형태인 직접 리소스부터 만들어 보자. 이 리소스는 매개변수가 전혀 없는 정적 유알아이를 가지며, 모든 문서 아이디의 목록을 한꺼번에 반환하는 역할을 한다. 마임 타입은 자바스크립트 객체 표기법(`application/json`) 으로 지정한다. 그 이유는 에스디케이가 함수 반환값(파이썬 리스트)을 자동으로 그 표기법으로 직렬화하여 클라이언트에 전송해 주기 때문이다. 강의노트 §2.2 의 팁 박스에서 정리한 대로, 마임 타입은 클라이언트가 응답을 어떻게 파싱할지를 결정하는 결정적 힌트 역할을 한다.

이번 셀에서 사용할 핵심 식별자를 정리해 두자. 데코레이터에는 두 가지 인자를 넣어야 한다. 첫째 인자는 유알아이 문자열이고, 둘째 인자는 마임 타입이다. 함수의 이름은 문서 목록을 반환한다는 의미를 담아 짓고, 반환 타입은 문자열 리스트로 선언한다. 에스디케이가 자동으로 직렬화해 주므로 별도의 변환 코드를 작성할 필요가 없다는 점이 매우 편리한 부분이다.

> [!tip] 유알아이의 스킴(scheme)은 자유롭게 정한다
> 유알아이의 스킴 부분은 도메인 의미를 담은 자유로운 접두사로 사용할 수 있다. 강의 예제에서는 문서 도메인이라는 의미를 담은 접두사를 쓰고 있지만, 실제 프로젝트에서는 자기 도메인에 맞춰 자유롭게 선택해도 된다. 예를 들어 한국 설계기준 데이터를 노출한다면 그것을 떠올리게 하는 접두사가 자연스러운 선택이 될 것이다. 마이다스 해석 결과를 노출한다면 그 시스템 이름을 담은 접두사가 적합하고, 빌딩 정보 모델링 데이터라면 그것을 가리키는 접두사도 좋은 선택이 된다. 의미를 명확하게 드러내는 이름일수록 노트북을 읽는 사람이 그 리소스의 정체를 직관적으로 이해하게 되므로 가독성이 좋아진다. 본 강의의 후반부 도메인 응용 트랙에서는 실제로 한국 설계기준 접두사를 사용하여 한국 설계기준을 노출하는 도메인 응용 예제를 다루게 된다.


In [ ]:
# Week_07.md §2.2 line ~939 — 고정 URI 리소스 정의 (cli_project/mcp_server.py:48-53)
# 이 셀은 단독 데모용. 다음 §4 에서 mcp_server.py 파일에 통합 저장된다.

demo_mcp = FastMCP('DocumentMCP-Demo', log_level='ERROR')

demo_docs = {
    'deposition.md': 'This deposition covers the testimony of Angela Smith, P.E.',
    'report.pdf':    'The report details the state of a 20m condenser tower.',
    'plan.md':       'The plan outlines the steps for the project implementation.',
}

@demo_mcp.resource(
    'docs://documents',
    mime_type='application/json',
)
def list_docs() -> list[str]:
    """모든 문서 ID 를 반환한다."""
    return list(demo_docs.keys())

# 고정 URI 리소스가 등록되었음을 확인
print('고정 URI 리소스 등록 완료: docs://documents -> list_docs')
print('샘플 출력 (실제 호출 시 SDK 가 JSON 으로 직렬화):', list_docs())


## §3. 매개변수가 들어간 형태의 리소스 만들기 — 단일 문서 내용 반환

이번에는 매개변수가 들어간 유알아이를 사용하여 단일 문서의 내용을 반환하는 리소스를 정의해 보자. 이 리소스의 핵심 동작 방식은 다음과 같다. 클라이언트가 특정 문서 아이디가 포함된 유알아이로 요청을 보내면, 에스디케이가 유알아이에서 매개변수 값을 자동으로 추출한다. 즉 문서 아이디라는 키워드 인자에 그 값을 채워서 서버 측 함수에 전달해 주는 것이다. 결과적으로 서버에서는 마치 그 인자를 직접 호출한 것처럼 처리되는 셈이다.

본 리소스의 핵심 식별자를 정리하면 다음과 같다. 데코레이터의 유알아이 부분에는 매개변수 자리표시자가 중괄호로 들어가야 하고, 마임 타입은 평문 텍스트로 지정한다. 함수의 이름은 단일 문서를 가져온다는 의미를 담아 짓는다. 한 가지 결정적으로 중요한 사항은, 유알아이의 자리표시자 이름과 함수 매개변수의 이름이 반드시 정확히 일치해야 한다는 점이다. 만약 매개변수 이름을 다르게 짓는다면 에스디케이가 자동 매핑을 수행할 수 없게 된다.

> [!finding] 마임 타입에 따른 분기 처리의 중요성
> 응답의 마임 타입은 클라이언트의 파싱 전략을 결정하는 결정적 단서가 된다. 즉 응답이 자바스크립트 객체 표기법 타입이라면 클라이언트가 그것을 파싱하여 파이썬 객체로 변환하고, 평문 타입이라면 별도의 파싱 처리 없이 문자열을 그대로 반환한다. 이렇게 마임 힌트가 명시적으로 주어지기 때문에, 다음 단원 §5 에서 작성할 클라이언트 측 분기 로직이 가능해진다. 즉 동일한 리소스 읽기 메서드 하나가 두 가지 서로 다른 응답 포맷을 모두 자동으로 처리할 수 있게 되는 것이다.


In [ ]:
# Week_07.md §2.2 line ~954 — 매개변수 URI 리소스 정의 (cli_project/mcp_server.py:57-64)

@demo_mcp.resource(
    'docs://documents/{doc_id}',
    mime_type='text/plain',
)
def fetch(doc_id: str) -> str:
    """특정 문서 ID 로 그 내용을 반환한다."""
    if doc_id not in demo_docs:
        raise ValueError(f'Doc with id {doc_id} not found')
    return demo_docs[doc_id]

# 매개변수 URI 리소스가 등록되었음을 확인
print('매개변수 URI 리소스 등록 완료: docs://documents/{doc_id} -> fetch')
# 직접 호출 (실제 흐름에서는 SDK 가 URI 파싱을 통해 자동 호출)
print('fetch("plan.md") 결과 =', fetch('plan.md'))


## §4. 서버 파일 업데이트 — 도구와 리소스를 모두 포함한 통합본 저장

앞선 노트북에서 이미 도구 두 개를 포함한 서버 파일을 만들어 두었다. 이번 단계에서는 그 파일에 §2 와 §3 에서 정의한 리소스 두 개를 추가하여, 도구와 리소스를 모두 포함한 완성본을 저장한다. 이렇게 만들어진 파일은 같은 폴더 안의 참조 구현과 동일한 구조를 가지게 된다. 즉 같은 폴더 안에 있는 참조 구현과 본 노트북의 결과물이 일치하는지를 비교하면서 학습하면 좋다.

단, 본 노트북에서는 프롬프트 컴포넌트는 아직 추가하지 않는다는 점을 미리 알아 두자. 프롬프트는 다음 노트북에서 본격적으로 다룰 예정이기 때문이다. 이렇게 단계적으로 컴포넌트를 하나씩 추가해 나가는 구성을 채택한 데에는 분명한 이유가 있다. 학생이 한 번에 모든 것을 접하지 않고, 하나의 컴포넌트씩 점진적으로 학습 부담을 늘려 가면서 각 컴포넌트의 역할을 분명히 체득하도록 의도된 설계인 것이다.

> [!action] 저장 후에 확인해야 할 사항
> 셀을 실행한 직후에 두 가지를 확인하면 좋다. 첫째, 운영체제 명령으로 파일 수정 시간을 점검하여 저장이 정상적으로 이루어졌는지를 본다. 둘째, 엠시피 인스펙터로 시각적으로 검증하고 싶다면 별도의 명령을 실행하면 된다. 그러면 인스펙터가 떠 오르고, 브라우저에 리소스 탭과 리소스 템플릿 탭이 보이게 된다. 그곳에서 각 리소스를 클릭하면 반환값과 마임 타입을 직접 점검할 수 있다.


In [ ]:
# Week_07.md §2.2 — mcp_server.py 통합 (도구 + 리소스). 다음 노트북 S6_05에서 프롬프트 추가.
# Source: cli_project/mcp_server.py (라인 1-65, prompt 제외)

server_code = '''from pydantic import Field
from mcp.server.fastmcp import FastMCP
from mcp.server.fastmcp.prompts import base

mcp = FastMCP("DocumentMCP", log_level="ERROR")


docs = {
    "deposition.md":   "This deposition covers the testimony of Angela Smith, P.E.",
    "report.pdf":      "The report details the state of a 20m condenser tower.",
    "financials.docx": "These financials outline the project\'s budget and expenditures.",
    "outlook.pdf":     "This document presents the projected future performance of the system.",
    "plan.md":         "The plan outlines the steps for the project\'s implementation.",
    "spec.txt":        "These specifications define the technical requirements for the equipment.",
}


# Tool: Read a doc
@mcp.tool(
    name="read_doc_contents",
    description="Read the contents of a document and return it as a string",
)
def read_document(
    doc_id: str = Field(description="ID of the document to read")
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    return docs[doc_id]


# Tool: Edit a doc
@mcp.tool(
    name="edit_document",
    description="Edit a document by replacing a string in the document content with a new string",
)
def edit_document(
    doc_id: str = Field(description="Id of the document that will be edited"),
    old_str: str = Field(description="The text to replace. Must match exactly, including white space"),
    new_str: str = Field(description="The text to insert in place of the old text"),
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    docs[doc_id] = docs[doc_id].replace(old_str, new_str)


# Resource: Direct — return all doc IDs
@mcp.resource(
    "docs://documents",
    mime_type="application/json",
)
def list_docs() -> list[str]:
    return list(docs.keys())


# Resource: Templated — return contents of a particular doc
@mcp.resource(
    "docs://documents/{doc_id}",
    mime_type="text/plain",
)
def fetch(doc_id: str) -> str:
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    return docs[doc_id]


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open('mcp_server.py', 'w', encoding='utf-8') as f:
    f.write(server_code)

print('mcp_server.py written with 2 tools + 2 resources.')
print('Next step: S6_05 will add the format prompt.')

## §5. 클라이언트에서 리소스 접근하기 — 리소스 읽기 메서드의 구현

이제 강의노트 §2.3 에서 약속한 내용을 코드로 옮길 차례다. 클라이언트의 리소스 읽기 메서드를 통해 서버에 정의된 리소스를 읽어 온다. 이때 가장 중요한 핵심은 응답의 마임 타입에 따른 분기 처리다. 응답의 마임 타입이 자바스크립트 객체 표기법이라면 그것을 파싱하여 파이썬 객체를 반환하고, 평문 텍스트라면 그대로 문자열을 반환한다.

이 기능을 정상적으로 동작시키려면 클라이언트 코드에 두 가지 추가 임포트가 필요하다. 첫 번째는 표준 라이브러리의 자바스크립트 객체 표기법 모듈이며, 이는 응답을 파이썬 객체로 파싱하기 위해 필요하다. 두 번째는 파이댄틱(pydantic) 라이브러리의 임의 유알(AnyUrl) 타입이며, 이 임포트는 유알아이 매개변수의 적절한 타입 처리를 보장해 주는 역할을 한다.

실제 메서드의 시그니처는 참조 구현의 클라이언트 파일에 정의되어 있다. 그 본질을 한국어로 풀어 보면 다음과 같이 요약할 수 있다. 먼저 세션의 리소스 읽기 메서드를 호출하여 응답을 받고, 그 응답에 포함된 콘텐츠 리스트에서 첫 번째 원소를 꺼낸다. 그다음 그 첫 번째 원소가 텍스트 리소스 콘텐츠인지를 타입 검사로 확인한다. 만약 그것이 텍스트 리소스 콘텐츠라면, 마임 타입이 자바스크립트 객체 표기법인 경우에는 그것을 파싱하여 반환하고, 그 외의 경우에는 텍스트를 별도 처리 없이 그대로 반환한다.

한 가지 더 짚어 둘 점이 있다. 응답의 콘텐츠 리스트는 항상 리스트 형태로 오지만, 일반적으로 첫 번째 원소만 사용하면 충분하다. 그 이유는 첫 원소 안에 실제 데이터(텍스트 필드)와 메타데이터(마임 타입 필드)가 함께 담겨 있기 때문이다. 강의노트 §2.3 의 표현을 빌리면 이런 식으로 정리할 수 있다. 리소스는 다양한 타입의 콘텐츠를 반환할 수 있으므로 클라이언트는 이를 적절히 파싱해야 하며, 그 파싱 분기를 본 메서드에서 일괄적으로 처리하는 것이다.

이렇게 만든 메서드의 한 가지 큰 장점은 호출자에게 응답 포맷을 노출하지 않는다는 점이다. 즉 이 메서드를 사용하는 애플리케이션 코드는 마임 타입이 무엇인지 신경 쓸 필요 없이 단순히 결과를 받아 사용하기만 하면 된다. 이런 식의 설계가 바로 깔끔한 추상화이며, 향후 새로운 마임 타입을 추가할 때도 호출 코드를 변경하지 않고 메서드 내부만 수정하면 되도록 만들어 준다.


In [ ]:
# Week_07.md §2.3 line ~1064 — MCPClient에 read_resource 추가, mcp_client.py 갱신
# Source: cli_project/mcp_client.py 전체 (라인 1-87, read_resource 포함)

client_code = '''import sys
import asyncio
import json
from pydantic import AnyUrl
from typing import Optional, Any
from contextlib import AsyncExitStack
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client


class MCPClient:
    def __init__(
        self,
        command: str,
        args: list[str],
        env: Optional[dict] = None,
    ):
        self._command = command
        self._args = args
        self._env = env
        self._session: Optional[ClientSession] = None
        self._exit_stack: AsyncExitStack = AsyncExitStack()

    async def connect(self):
        server_params = StdioServerParameters(
            command=self._command,
            args=self._args,
            env=self._env,
        )
        stdio_transport = await self._exit_stack.enter_async_context(
            stdio_client(server_params)
        )
        _stdio, _write = stdio_transport
        self._session = await self._exit_stack.enter_async_context(
            ClientSession(_stdio, _write)
        )
        await self._session.initialize()

    def session(self) -> ClientSession:
        if self._session is None:
            raise ConnectionError(
                "Client session not initialized. Call connect first."
            )
        return self._session

    async def list_tools(self) -> list[types.Tool]:
        result = await self.session().list_tools()
        return result.tools

    async def call_tool(
        self, tool_name: str, tool_input: dict
    ) -> types.CallToolResult | None:
        return await self.session().call_tool(tool_name, tool_input)

    async def list_prompts(self) -> list[types.Prompt]:
        # NOTE: stub — implemented in S6_05
        result = await self.session().list_prompts()
        return result.prompts

    async def get_prompt(self, prompt_name, args: dict[str, str]):
        # NOTE: stub — implemented in S6_05
        result = await self.session().get_prompt(prompt_name, args)
        return result.messages

    async def read_resource(self, uri: str) -> Any:
        # Week_07.md §2.3 — MIME-aware parsing
        result = await self.session().read_resource(AnyUrl(uri))
        resource = result.contents[0]
        if isinstance(resource, types.TextResourceContents):
            if resource.mimeType == "application/json":
                return json.loads(resource.text)
            return resource.text

    async def cleanup(self):
        await self._exit_stack.aclose()
        self._session = None

    async def __aenter__(self):
        await self.connect()
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        await self.cleanup()
'''

with open('mcp_client.py', 'w', encoding='utf-8') as f:
    f.write(client_code)

print('mcp_client.py written with read_resource (MIME-aware).')

## §6. 인스펙터와 코드 — 두 가지 검증 경로 중 무엇을 쓸 것인가

리소스가 의도한 대로 동작하는지를 검증하는 방법은 크게 두 가지가 있다. 두 가지 방식을 모두 알아 두면, 상황에 맞는 적절한 검증 경로를 선택해서 사용할 수 있다.

**첫 번째 방법: 엠시피 인스펙터를 사용한 시각적 검증**:

먼저 인스펙터를 실행하는 명령을 실행하여 인스펙터를 띄운다. 그다음 브라우저에서 리소스 탭과 리소스 템플릿 탭을 차례로 클릭하여 각 리소스를 시각적으로 확인할 수 있다. 강의노트 §2.2 에 첨부된 스크린샷에서 보이는 그 화면이 바로 이 방식의 결과물이다. 이 방식은 빠르게 한눈에 확인하고 싶을 때 유용하다. 또한 새 리소스를 만드는 도중에 결과를 즉시 살펴보고 싶을 때도 편리한 방식이다.

**두 번째 방법: 파이썬 코드를 사용한 자동화 검증**:

두 번째 방식은 클라이언트의 비동기 컨텍스트 안에서 리소스 읽기 메서드를 호출하여 응답을 검증하는 코드 기반 접근이다. 이 방식의 가장 큰 장점은 동일한 검증 로직을 단위 테스트나 지속적 통합 파이프라인에 그대로 옮겨 쓸 수 있다는 점이다. 즉 리소스가 정상 동작하는지를 자동화된 테스트로 보장할 수 있게 된다는 의미다.

본 노트북에서는 두 번째인 코드 경로에 집중한다. 다음 셀에서 클라이언트를 띄우고 직접 리소스와 매개변수 리소스 두 가지를 모두 호출하여, 두 응답이 모두 의도한 형태로 반환되는지를 직접 확인해 보겠다.


In [ ]:
# Week_07.md §2.3 line ~1133 — async with MCPClient: read_resource 호출
import sys
if 'mcp_client' in sys.modules:
    del sys.modules['mcp_client']
from mcp_client import MCPClient

import os
USE_UV = os.getenv('USE_UV', '0') == '1'
command, args = ('uv', ['run', 'mcp_server.py']) if USE_UV else ('python', ['mcp_server.py'])

async def demo_read_resource():
    async with MCPClient(command=command, args=args) as client:
        # Direct resource: returns list[str], JSON-parsed by client
        doc_ids = await client.read_resource('docs://documents')
        print('--- docs://documents (application/json) ---')
        print(type(doc_ids).__name__, '->', doc_ids)
        print()

        # Templated resource: returns string (text/plain)
        content = await client.read_resource('docs://documents/plan.md')
        print('--- docs://documents/plan.md (text/plain) ---')
        print(type(content).__name__, '->', content)

await demo_read_resource()

## §7. 골뱅이 멘션 시뮬레이션 — 사용자 의도가 프롬프트에 주입되는 흐름

강의노트 §2.3 (라인 약 1150 부터 1156 까지)에는 골뱅이 멘션 워크플로 체크리스트가 여섯 단계로 정리되어 있다. 본 셀에서는 그 흐름의 핵심을 코드로 압축하여 재현해 본다. 단계별로 정리하면 다음과 같다.

첫 번째 단계는 토큰 탐지다. 사용자 입력 버퍼에서 골뱅이 토큰을 정규식으로 탐지한다. 두 번째 단계는 리소스 페치다. 멘션된 각 문서 아이디마다 매개변수 유알아이를 만들어 리소스 읽기 메서드로 내용을 가져온다. 세 번째 단계는 프롬프트 주입이다. 가져온 텍스트 내용을 문서 태그로 감싸서 프롬프트의 앞부분에 삽입한다. 네 번째 단계는 메시지 전송이다. 마지막으로 사용자 메시지를 그대로 클로드에게 전송한다.

> [!finding] 왜 리소스 방식이 도구 호출보다 효율적인가
> 만약 도구 호출 방식으로 동일한 동작을 구현했다면 어떻게 될까 생각해 보자. 클로드가 먼저 문서 읽기 도구를 호출해야겠다고 결정하는 한 번의 언어모델 왕복이 추가로 발생한다. 그러나 리소스 방식에서는 사용자가 골뱅이 기호로 의도를 이미 선언한 상태이므로, 첫 메시지에 내용이 이미 포함된 채로 클로드에게 전달된다. 즉 추가 왕복이 0회다. 사용자의 의도가 명확한 직접 파일 참조 시나리오에서는 리소스 방식이 훨씬 빠르다는 것이 강의노트 §2.3 의 핵심 메시지다.


In [ ]:
# Week_07.md §2.3 line ~1133 — @멘션 시뮬레이션
import re

# 사용자가 두 문서를 동시에 멘션한 입력 예시
user_input = 'Please summarize @plan.md and compare it with @report.pdf.'
mentions = re.findall(r'@(\S+)', user_input)
print('탐지된 멘션 목록:', mentions)
print()

async def inject_mentions(user_input: str):
    """사용자 입력에서 @멘션을 추출해 리소스 내용을 프롬프트에 주입한다."""
    mentions = re.findall(r'@(\S+)', user_input)
    async with MCPClient(command=command, args=args) as client:
        injected_blocks = []
        for doc_id in mentions:
            content = await client.read_resource(f'docs://documents/{doc_id}')
            injected_blocks.append(
                f'<document id="{doc_id}">\n{content}\n</document>'
            )
        injected = '\n'.join(injected_blocks)
        final_prompt = f'{injected}\n\n{user_input}'
        return final_prompt

final = await inject_mentions(user_input)
print('--- Claude 에게 실제로 전송될 최종 프롬프트 ---')
print(final)


## §8. 마임 타입 분기 시연 — 자바스크립트 객체 표기법과 평문의 자동 분기

이번 셀에서는 클라이언트 리소스 읽기 메서드의 분기 로직이 실제로 의도한 대로 동작하는지를 직접 검증해 본다. 두 가지 케이스를 비교하면서 살펴보자.

첫 번째 케이스는 직접 리소스다. 이 리소스의 마임 타입은 자바스크립트 객체 표기법으로 지정되어 있으므로, 클라이언트는 응답을 받자마자 파싱 함수로 파싱을 수행한다. 그 결과로 파이썬의 문자열 리스트 객체가 반환된다. 즉 호출자 입장에서는 마치 처음부터 파이썬 리스트를 받은 것처럼 사용할 수 있게 된다는 뜻이다.

두 번째 케이스는 매개변수 리소스다. 이 리소스의 마임 타입은 평문 텍스트이므로, 클라이언트는 별도의 파싱 처리 없이 응답을 그대로 문자열 객체로 반환한다. 즉 추가적인 변환 단계가 필요 없는 가장 단순한 흐름이라고 할 수 있다.

결과적으로 단일 메서드 하나가 두 가지 응답 포맷을 모두 자동으로 처리한다. 그 덕분에 호출자(즉 애플리케이션 로직)는 마임 타입을 직접 확인하거나 응답 변환 코드를 별도로 작성할 필요가 없어진다. 이것이 바로 강의노트 §2.3 에서 강조한 관심사 분리의 효과다. 즉 엠시피 클라이언트는 서버와의 통신과 응답 파싱을 담당하고, 애플리케이션 로직은 그 데이터를 어떻게 활용할지에만 집중하면 되도록 책임이 깔끔하게 나뉘어 있는 것이다. 이런 식의 추상화 덕분에 향후 새로운 마임 타입이 추가되더라도 호출 코드를 변경하지 않고 메서드 내부만 수정하면 되도록 만들어 준다는 부수적인 이점도 있다.


In [ ]:
# Week_07.md §2.3 line ~1075-1087 — MIME 분기 자동 검증

async def verify_mime_branching():
    """두 리소스의 응답 타입이 MIME 에 맞게 분기되는지 검증한다."""
    async with MCPClient(command=command, args=args) as client:
        # 고정 URI -> JSON 파싱 후 list 반환
        result_json = await client.read_resource('docs://documents')
        assert isinstance(result_json, list), '기대 타입: list (JSON 응답)'
        print(f'OK: docs://documents -> list, 항목 수 {len(result_json)}')

        # 매개변수 URI -> 평문 그대로 str 반환
        result_text = await client.read_resource('docs://documents/spec.txt')
        assert isinstance(result_text, str), '기대 타입: str (text/plain 응답)'
        print(f'OK: docs://documents/spec.txt -> str, 길이 {len(result_text)}')

        return result_json, result_text

await verify_mime_branching()


## §9. 다음 단계로 — 무엇을 만들었고 다음에 무엇을 더할 것인가

본 노트북에서 완성한 자산을 정리하면 다음과 같다. 첫 번째 자산은 서버 파일의 완성본이다. 이 파일은 이제 도구 두 개와 리소스 두 개를 모두 포함하게 되었다. 도구는 기존의 문서 읽기 도구와 문서 편집 도구 두 개이고, 리소스는 직접 형태의 문서 목록 리소스와 매개변수 형태의 단일 문서 리소스 두 개다. 두 번째 자산은 클라이언트 파일의 확장이다. 이 파일에 리소스 읽기 메서드가 새로 추가되었으며, 이 메서드는 응답의 마임 타입에 따라 자동으로 분기 처리를 수행하도록 만들어졌다. 세 번째 자산은 골뱅이 멘션 시뮬레이션 함수다. 사용자 입력에서 문서 아이디를 추출하여 자동으로 프롬프트에 주입하는 흐름을 직접 구현해 보았다.

**다음 노트북에서 다룰 내용**:

다음 노트북에서는 엠시피의 세 번째 컴포넌트인 프롬프트를 본격적으로 학습한다. 구체적으로 어떤 내용을 다루게 되는지 정리해 보자. 우선 프롬프트 데코레이터를 사용하여 포맷팅 프롬프트를 정의하는 방법을 익히게 된다. 그다음 사용자 메시지와 어시스턴트 메시지 두 가지 타입을 혼합하여 적은 예시 학습 스타일의 프롬프트를 구성하는 패턴을 다룬다. 또한 클라이언트 측에서는 프롬프트 목록 조회와 프롬프트 가져오기 두 메서드를 구현하여, 서버가 제공하는 프롬프트 카탈로그를 조회하고 활용할 수 있게 만든다. 마지막으로 슬래시 명령을 파싱하여 디스패치하는 시뮬레이션을 진행한다. 이를 통해 도구와 리소스와 프롬프트라는 세 컴포넌트가 한 사이클 안에서 어떻게 서로 협업하는지를 종합적으로 시연하게 된다.

강의노트 §2.4 와 §2.5 (라인 약 1166 부터 1395 까지) 가 다음 노트북의 텍스트 소스로 사용된다. 본 노트북과 마찬가지로 강의노트의 핵심 문장과 박스 인용을 직접 가져와 한국어 본문에 녹여 넣는 방식으로 구성된다.

> [!ref] 강의노트 §2.4·§2.5 (라인 약 1166-1395) → 다음 노트북
